In [3]:
import pandas as pd
from pathlib import Path
from ollama import Client
import json
from time import sleep
import asyncio
import json
from ollama import AsyncClient

In [4]:

base_path = Path('/Volumes/X9 Pro/newspapers-csv/lwm-csv')
file_name = '0002610.csv'

In [5]:
df = pd.read_csv(base_path / file_name, index_col=0)

In [6]:
df.head(3)

,article_headline,item_type,ocr_quality_mean,ocr_quality_sd,word_count,plain_text_file,date,newspaper_title,location,source,text,year,month,day,NLP,issue,art_num
0002610/1905/0819/0002610_19050819_art0062_metadata.xml,LUNCHEON.,ARTICLE,0.9785,0.0707,104,0002610_19050819_art0062.txt,1905-08-19,"The St. Helens Examiner, and Prescot Weekly News.","Saint Helens, Merseyside, England",British Library Living with Machines Project,THE LUNCHEON.\n\nAt the invitation or the pres...,1905,8,8,2610,50819,art0062
0002610/1905/0819/0002610_19050819_art0069_metadata.xml,NaN,ARTICLE,0.8125,0.2073,4,0002610_19050819_art0069.txt,1905-08-19,"The St. Helens Examiner, and Prescot Weekly News.","Saint Helens, Merseyside, England",British Library Living with Machines Project,la\n\n193 128.\n193\n,1905,8,8,2610,50819,art0069
0002610/1905/0819/0002610_19050819_art0055_metadata.xml,"Examiner, Saturday, August 19, 1905.",ARTICLE,0.9817,0.0784,527,0002610_19050819_art0055.txt,1905-08-19,"The St. Helens Examiner, and Prescot Weekly News.","Saint Helens, Merseyside, England",British Library Living with Machines Project,"The Examiner, Saturday, August 19, 1905.\n\nAp...",1905,8,8,2610,50819,art0055


In [7]:
df['text_length'] = df['text'].str.len()

In [8]:
df.year.min(),df.year.max()

(np.int64(1879), np.int64(1920))

In [9]:
df_ocr = df[(df.ocr_quality_mean > 0.95) & (df.text_length > 1000)]

In [10]:
out_path = Path('data')
out_path.mkdir(exist_ok=True)
df_ocr_train = df_ocr.sample(frac=0.8, random_state=42)
df_ocr_test = df_ocr.drop(df_ocr_train.index)
df_ocr_train.to_csv(out_path / 'train.csv', index=False)
df_ocr_test.to_csv(out_path / 'test.csv', index=False)
df_ocr_train.shape, df_ocr_test.shape

((63202, 18), (15800, 18))

In [11]:
df_ocr_train_sample = df_ocr_train.sample(1000, random_state=42)
df_ocr_train_sample.to_csv(out_path / 'train_sample.csv', index=False)

In [12]:
# three prompt types
# guess the year of publication
# guess the year of publication with reasoning
# write a an article given the year of publication
# write an article given the year of publication with reasoning

In [13]:
with open('api_key.json') as f:
    api_key = json.load(f)['api_key']


import asyncio
import json
from ollama import AsyncClient

import asyncio
import json
from ollama import AsyncClient
from tqdm import tqdm


async def prompt_ollama(messages, client, model="llama3", max_retries=5):

    for attempt in range(max_retries):
        try:
            response = await client.chat(
                model=model,
                messages=messages
            )
            #print(f"Response: {response}")
            return response["message"]["content"]

        except Exception as e:
            print(f"Retry {attempt+1}/{max_retries}: {e}")
            await asyncio.sleep(0.5)

    return None


async def process_messages(
    message_list,
    model="llama3",
    outfile="results.jsonl",
    concurrency=10,
    ):

    client = AsyncClient(host="https://ollama.com",
                        headers={'Authorization': api_key})
    semaphore = asyncio.Semaphore(concurrency)

    async def worker(messages):

        async with semaphore:

            response = await prompt_ollama(messages, client, model)

            record = {
                "messages": messages,
                "response": response
            }

            with open(outfile, "a", encoding="utf-8") as f:
                f.write(json.dumps(record, ensure_ascii=False) + "\n")

            return record


    tasks = [worker(messages) for messages in message_list]

    with tqdm(total=len(tasks)) as pbar:

        for future in asyncio.as_completed(tasks):

            await future
            pbar.update(1)

In [14]:
import re
import unicodedata


def normalize_ocr_text(text: str, max_tokens: int = -1) -> str:
    """
    Clean and normalize OCR-style noisy text.

    Steps:
    - Normalize unicode
    - Remove hyphenated line breaks
    - Remove stray line breaks inside paragraphs
    - Collapse whitespace
    - Remove obvious OCR garbage characters
    """

    # Normalize unicode characters
    text = unicodedata.normalize("NFKC", text)

    # Fix hyphenated line breaks: "sup-\nplies" -> "supplies"
    text = re.sub(r"-\n\s*", "", text)

    # Replace remaining line breaks with spaces
    text = re.sub(r"\n+", " ", text)

    # Remove weird isolated punctuation artifacts
    text = re.sub(r"[•~\\]+", " ", text)

    # Remove stray quotes
    text = text.replace("’", "'").replace("`", "'")

    # Remove multiple punctuation artifacts
    text = re.sub(r"[\"']{2,}", "'", text)

    # Remove random standalone characters
    text = re.sub(r"\b[a-zA-Z]\b", "", text)

    # Collapse multiple spaces
    text = re.sub(r"\s{2,}", " ", text)

    # Max tokens (for LLM input limits)
    tokens = text.split()
    
    text = " ".join(tokens[:max_tokens])

    # Trim
    text = text.strip()

    return text

In [16]:
messages_list = [
    [
    {"role": "system", "content": "You are a helpful editor creates an index by generative abtractive keyphrases from historical newspaper articles. The article is demarcated by triple hashtags. Abstractive key phrases are single words or longer phrases that describe the main topics, entities, and themes of the article. Return abstractive key phrases a JSON list of strings."},
    {"role": "user", "content": f"Extract ten key phrases from the following article: \n\n ### {normalize_ocr_text(row.text)[:1000]} ###. Only return abstractive key phrases a JSON list of strings."}]
        for _, row in df_ocr_train_sample.iterrows()
]

In [17]:
messages_list[10]

[{'role': 'system',
  'content': 'You are a helpful editor creates an index by generative abtractive keyphrases from historical newspaper articles. The article is demarcated by triple hashtags. Abstractive key phrases are single words or longer phrases that describe the main topics, entities, and themes of the article. Return abstractive key phrases a JSON list of strings.'},
 {'role': 'user',
  'content': "Extract ten key phrases from the following article: \n\n ### VOLUNTEER ORDERS. 'Sunday (to-morrow).—Class; Firing at Altear. Parade, 8.15 .., outside headquarters or Central Station, 8.40 .. Dress: Musketry Order; one ration to be carried. Monday.-7.15 .., Visional Training; 8.15 , .., Dayonet Fighting. Thursclay.7-7.15 .. and 8.15 .., Entrenching and Field Work. Orderly.—Corporal McDermid. . WARBURTON, Officer Commanding. Missionary Association.—The monthly meeting of the Missionary Association was held in the Parish Room on Monday evening, and was presided over by the Rev. . Colli

In [18]:
await process_messages(messages_list, model="deepseek-v3.1:671b-cloud", outfile="key_phrases.jsonl", concurrency=5)

  1%|          | 12/1000 [00:25<34:37,  2.10s/it]


CancelledError: 

In [19]:
keyphrases = []
with open("key_phrases.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        keyphrases.append(record)

In [21]:
keyphrases[0]

{'messages': [{'role': 'system',
   'content': 'You are a helpful editor creates an index by generative abtractive keyphrases from historical newspaper articles. The article is demarcated by triple hashtags. Abstractive key phrases are single words or longer phrases that describe the main topics, entities, and themes of the article. Return abstractive key phrases a JSON list of strings.'},
  {'role': 'user',
   'content': 'Extract ten key phrases from the following article: \n\n ### "ABSOLUTELY HOPELESS." Councillor\' Strong Comments On The Telephone System. motion to the effect "That this Council are of opinion that the telephone system as at .present administered is utterly inadequate for modern business requirements and constitutes grave menace to the after-war development of commerce, and that His Majesty\' Government be urged to take steps to remedy the same," was Moved by Councillor Pemberton at the conclusion of the ordinary business at the Town Council meeting on Tuesday. Every

In [23]:
df_ocr_train_sample.iloc[0]

article_headline                                Iron and Steel Trade.
item_type                                                     ARTICLE
ocr_quality_mean                                               0.9864
ocr_quality_sd                                                 0.0564
word_count                                                        546
plain_text_file                          0002610_19120217_art0145.txt
date                                                       1912-02-17
newspaper_title     The St. Helens Examiner, and Prescot Weekly News.
location                            Saint Helens, Merseyside, England
source                   British Library Living with Machines Project
text                The Iron and Steel Trade.\n\nMr. Harold Smith'...
year                                                             1912
month                                                               2
day                                                                 2
NLP                 

In [41]:
keyphrases[0]['response']

'Here is the JSON list of abstractive keyphrases derived from the provided text:\n\n```json\n[\n  "telephone system inadequacy",\n  "post-war commerce development",\n  "government inaction",\n  "business frustration",\n  "public inconvenience",\n  "criticism of administration",\n  "reconstruction delays",\n  "call for reform"\n]\n```'

In [45]:
import json
import re

def parse_llm_json(output: str):
    """
    Parse LLM output that may contain JSON either as plain text
    or inside ```json ... ``` code fences.

    Args:
        output (str): The text returned by the LLM.

    Returns:
        Parsed JSON as Python object (dict, list, etc.)
    """

    # Try to find ```json ... ``` block
    code_fence_match = re.search(r"```json\s*(.*?)```", output, re.DOTALL | re.IGNORECASE)
    
    if code_fence_match:
        json_str = code_fence_match.group(1).strip()
    else:
        # Fallback: assume the whole text is JSON
        json_str = output.strip()

    try:
        return json.loads(json_str)
    except json.JSONDecodeError:
        # Optional: fallback to safer literal_eval (only for lists/dicts)
        import ast
        try:
            return ast.literal_eval(json_str)
        except Exception as e:
            print(f"Failed to parse JSON from LLM output: {e}\nOutput was:\n{output}")
            return []
            #raise ValueError(f"Failed to parse JSON from LLM output: {e}\nOutput was:\n{output}")


In [49]:
messages_list = [
    [
    {"role": "system", "content": "You are a helpful journalist and historian that generates a newspaper article given a date of publication and key phrases. Make sure the text is historically accurate by paying attention to the date of publication."},
    {"role": "user", "content": f"Generate an article published in {df.iloc[i]['year']} given the key phrases {parse_llm_json(kp['response'])}"},
    {"role": "assistant", "content": kp['messages'][1]['content'].split("###")[1]}]
        for i, kp in enumerate(keyphrases)
]

Failed to parse JSON from LLM output: unexpected character after line continuation character (<unknown>, line 1)
Output was:
\[
"key phrases extraction",
"stack drying problem",
"six millions annual loss",
"artificial heat application",
"self-curing hay",
"history of agricultural innovation",
"Plain Facts pamphlet",
"Mr. Gibbs' process",
"fever of expectation",
"merits and demerits"
]
Failed to parse JSON from LLM output: invalid character '’' (U+2019) (<unknown>, line 3)
Output was:


## Council investigates Council investigates Council investigates Council investigates Council investigates Council investigates Council investigates Council investigates Council investigates Council investigates Council investigates Council investigates Council investigates Council investigates Council investigates Council investigates Council investigates Council ��The Council responded confidently in their response.

The Commission’s examination assessed that Council’s investigation and confirmations.

In [52]:
messages_list = [m for m in messages_list if m[2]['content']]
len(messages_list)

319

In [51]:
system_year_prediction = "You are a helpful assistant that can guess the year of publication of a newspaper article."
system_write = "You are a helpful assistant that can write an article given the year of publication and ten keywords." 
reasoning_tag = "You can also provide reasoning for your writings."